# E-commerce Customer Journey & Conversion Analysis

## Sprint 3 — Data Cleaning & Preparation

### Objective

The objective of this notebook is to transform the raw e-commerce behavioral data into a clean, consistent, and analysis-ready dataset.

Based on the findings from the Data Understanding stage, this sprint will focus on:

- Combining the five monthly datasets into a consistent processing workflow.
- Converting `event_time` into an appropriate datetime format.
- Removing exact duplicate records.
- Evaluating and treating missing values in `category_code`, `brand`, and `user_session`.
- Defining a documented treatment strategy for zero and negative prices.
- Preserving identifier fields appropriately.
- Creating analytical features required for customer journey, purchasing behavior, and cohort analysis.
- Optimizing the processed dataset for efficient downstream analysis.

All cleaning decisions will be documented to preserve transparency and reproducibility.

In [64]:
%pip install pyarrow

   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.9 MB 653.6 kB/s eta 0:00:43
   ---------------------------------------- 0.0/27.9 MB 653.6 kB/s eta 0:00:43
   ---------------------------------------- 0.1/27.9 MB 469.7 kB/s eta 0:01:00
   ---------------------------------------- 0.1/27.9 MB 438.1 kB/s eta 0:01:04
   ---------------------------------------- 0.1/27.9 MB 481.4 kB/s eta 0:00:58
   ---------------------------------------- 0.1/27.9 MB 448.2 kB/s eta 0:01:02
   ---------------------------------------- 0.2/27.9 MB 525.1 kB/s eta 0:00:53
   ---------------------------------------- 0.2/27.9 MB 525.1 kB/s eta 0:00:53
   ---------------------------------------- 0.2/27.9 MB 461.0 kB/s eta 0:01:00
   ---------------------------------------- 0.2/27.9 MB 465.5 kB/s eta 0:01:00
   ---------------------------------------- 0.3/27.9 MB 477.2 kB/s eta


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\Sweet\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [65]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path


In [6]:
raw_data_path = Path("../data/raw")
processed_data_path = Path("../data/processed")

csv_files = sorted(raw_data_path.glob("*.csv"))

In [7]:
for file in csv_files:
    print(file.name)

2019-Dec.csv
2019-Nov.csv
2019-Oct.csv
2020-Feb.csv
2020-Jan.csv


In [8]:
month_order ={
    "2019-Oct.csv": 1,
    "2019-Nov.csv": 2,
    "2019-Dec.csv": 3,
    "2020-Jan.csv": 4,
    "2020-Feb.csv": 5
}

csv_files = sorted(
    csv_files,
    key=lambda file: month_order[file.name]
)

for file in csv_files:
    print (file.name)

2019-Oct.csv
2019-Nov.csv
2019-Dec.csv
2020-Jan.csv
2020-Feb.csv


In [9]:
test_chuck = pd.read_csv(csv_files[0], nrows=5)

test_chuck["event_time"]

0    2019-10-01 00:00:00 UTC
1    2019-10-01 00:00:03 UTC
2    2019-10-01 00:00:07 UTC
3    2019-10-01 00:00:07 UTC
4    2019-10-01 00:00:15 UTC
Name: event_time, dtype: str

The data type should be datetime64 because the variable represents both date and time.

In [10]:
test_chuck["event_time"] = pd.to_datetime(test_chuck["event_time"], utc=True)

test_chuck["event_time"].dtype


datetime64[us, UTC]

In [11]:
test_chuck["event_time"] = pd.to_datetime(test_chuck["event_time"], utc=True, errors="coerce")

test_chuck["event_time"].isna().sum()

np.int64(0)

In [12]:
invalid_event_times = 0

In [13]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["event_time"]
    ):
          parsed_event_time = pd.to_datetime(
            chunk["event_time"],
            utc=True,
            errors="coerce"
        )

          invalid_event_times += parsed_event_time.isna().sum()
        

In [14]:
invalid_event_times

np.int64(0)

### Event Time Cleaning

The `event_time` column was originally stored as a string despite representing timestamp information.

A full-dataset validation was performed by parsing all event timestamps using a timezone-aware UTC datetime representation. No invalid timestamps were detected during the conversion.

Therefore, `event_time` will be converted to a UTC datetime data type during data preparation.

**Cleaning decision:**

- Convert `event_time` from string to timezone-aware datetime.
- Preserve UTC as the reference timezone.
- No records need to be removed due to invalid timestamps.

In [15]:
duplicate_event_counts = pd.Series(dtype="int64")

In [16]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000
    ):
        duplicate_mask = chunk.duplicated(
            keep="first"
        )

        duplicate_events = chunk.loc[
            duplicate_mask,
            "event_type"
        ].value_counts()

        duplicate_event_counts = duplicate_event_counts.add(
            duplicate_events,
            fill_value=0
        )

In [17]:
print(
    duplicate_event_counts.sort_values(ascending=False)
)

print("Duplicates detected within chunks",int(duplicate_event_counts.sum()))

event_type
remove_from_cart    991817.0
cart                115342.0
view                  1031.0
purchase               905.0
dtype: float64
Duplicates detected within chunks 1109095


If all exact duplicate records were removed without further investigation, legitimate repeated user interactions could potentially be discarded, which could bias the resulting funnel and behavioral analysis.

In [18]:
duplicate_remove_examples = pd.DataFrame()

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000
    ):
        duplicate_mask = (
            chunk.duplicated(keep=False)
            & (chunk["event_type"] == "remove_from_cart")
        )

        if duplicate_mask.any():
            duplicate_remove_examples = chunk.loc[
                duplicate_mask
            ].head(20)

            break

    if not duplicate_remove_examples.empty:
        break

duplicate_remove_examples


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
12,2019-10-01 00:00:30 UTC,remove_from_cart,5809103,1487580013522845895,NaN,irisk,0.60,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
13,2019-10-01 00:00:30 UTC,remove_from_cart,5809103,1487580013522845895,NaN,irisk,0.60,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
29,2019-10-01 00:01:05 UTC,remove_from_cart,5858981,1487580007634043851,NaN,de.lux,0.79,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
30,2019-10-01 00:01:05 UTC,remove_from_cart,5858981,1487580007634043851,NaN,de.lux,0.79,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
34,2019-10-01 00:01:07 UTC,remove_from_cart,5312,1487580009471148064,NaN,runail,1.27,467916806,2f5b5546-b8cb-9ee7-7ecd-84276f8ef486
36,2019-10-01 00:01:07 UTC,remove_from_cart,5312,1487580009471148064,NaN,runail,1.27,467916806,2f5b5546-b8cb-9ee7-7ecd-84276f8ef486
37,2019-10-01 00:01:10 UTC,remove_from_cart,5823915,1487580007432717250,NaN,milv,1.59,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
38,2019-10-01 00:01:10 UTC,remove_from_cart,5823915,1487580007432717250,NaN,milv,1.59,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
50,2019-10-01 00:01:24 UTC,remove_from_cart,5664641,1487580011425693811,NaN,NaN,0.70,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
51,2019-10-01 00:01:24 UTC,remove_from_cart,5664641,1487580011425693811,NaN,NaN,0.70,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9


In [19]:
duplicate_group_sizes = (
    duplicate_remove_examples
    .groupby(
        list(duplicate_remove_examples.columns),
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
)

duplicate_group_sizes.head(10)

event_time               event_type        product_id  category_id          category_code  brand     price  user_id    user_session                        
2019-10-01 00:00:30 UTC  remove_from_cart  5809103     1487580013522845895  NaN            irisk     0.60   429681830  49e8d843-adf3-428b-a2c3-fe8bc6a307c9    2
2019-10-01 00:01:05 UTC  remove_from_cart  5858981     1487580007634043851  NaN            de.lux    0.79   429681830  49e8d843-adf3-428b-a2c3-fe8bc6a307c9    2
2019-10-01 00:01:07 UTC  remove_from_cart  5312        1487580009471148064  NaN            runail    1.27   467916806  2f5b5546-b8cb-9ee7-7ecd-84276f8ef486    2
2019-10-01 00:01:10 UTC  remove_from_cart  5823915     1487580007432717250  NaN            milv      1.59   429681830  49e8d843-adf3-428b-a2c3-fe8bc6a307c9    2
2019-10-01 00:01:24 UTC  remove_from_cart  5664641     1487580011425693811  NaN            NaN       0.70   429681830  49e8d843-adf3-428b-a2c3-fe8bc6a307c9    2
2019-10-01 00:01:38 UTC  remove_from_ca

In [20]:
first_chunk = pd.read_csv(
    csv_files[0],
    nrows=500000
)

remove_duplicates = first_chunk[
    (first_chunk["event_type"] == "remove_from_cart")
    & first_chunk.duplicated(keep=False)
]

remove_duplicate_sizes = (
    remove_duplicates
    .groupby(
        list(remove_duplicates.columns),
        dropna=False
    )
    .size()
)

remove_duplicate_sizes.value_counts().sort_index()

2     19143
3       220
4        93
5        21
6        10
7         8
8        14
9         3
10        1
14        1
Name: count, dtype: int64

Although most duplicate groups appear twice, some identical remove_from_cart events occur multiple times. Since the dataset does not include quantity information or additional event-level identifiers, there is insufficient evidence to classify all exact duplicates as data errors. Removing them could discard legitimate repeated user interactions and bias the behavioral analysis.

In [21]:
most_repeated_event = (
    remove_duplicates
    .groupby(
        list(remove_duplicates.columns),
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
    .head(10)
)

most_repeated_event

event_time               event_type        product_id  category_id          category_code  brand    price  user_id    user_session                        
2019-10-02 11:36:03 UTC  remove_from_cart  5873241     1487580004857414477  NaN            lianail  6.33   200698039  359ea082-0dba-43d2-9b68-aa0a15be20db    14
2019-10-03 12:43:03 UTC  remove_from_cart  5724563     1487580005427839846  NaN            irisk    2.48   526955800  d86b3bbc-2aa7-70f5-7da9-a41bbfc4bb73    10
2019-10-03 18:50:16 UTC  remove_from_cart  5773377     1487580005134238553  NaN            runail   2.78   440617826  6e468039-996b-4b75-ba7b-4db73695b52a     9
2019-10-04 06:42:52 UTC  remove_from_cart  5784039     1487580006551913373  NaN            NaN      3.02   436498465  cd019990-1035-44cc-9753-6191ebaa71f9     9
2019-10-03 18:55:33 UTC  remove_from_cart  5814042     1783999063314661546  NaN            grattol  10.32  526615137  8570e500-9575-4863-9862-913e504b5859     9
2019-10-03 18:55:55 UTC  remove_from_car

In [22]:
user_context = first_chunk[
    (first_chunk["user_id"] == 200698039)
    & (first_chunk["user_session"] == "359ea082-0dba-43d2-9b68-aa0a15be20db")
]

user_context[
    [
        "event_time",
        "event_type",
        "product_id",
        "brand",
        "price"
    ]
].sort_values("event_time")

,event_time,event_type,product_id,brand,price
209305,2019-10-02 09:32:07 UTC,remove_from_cart,5565822,NaN,0.71
209309,2019-10-02 09:32:07 UTC,remove_from_cart,5565822,NaN,0.71
209412,2019-10-02 09:32:49 UTC,cart,5712801,NaN,3.02
209426,2019-10-02 09:32:56 UTC,cart,6730,NaN,7.92
209436,2019-10-02 09:33:01 UTC,cart,5620924,NaN,6.65
...,...,...,...,...,...
250190,2019-10-02 12:59:28 UTC,purchase,5848901,bpw.style,1.27
250191,2019-10-02 12:59:28 UTC,purchase,5712801,NaN,3.02
250192,2019-10-02 12:59:28 UTC,purchase,5706142,irisk,0.63
250193,2019-10-02 12:59:28 UTC,purchase,5848909,bpw.style,0.79


In [23]:
product_context = user_context[
    user_context["product_id"] == 5873241
]

product_context[
    [
        "event_time",
        "event_type",
        "product_id",
        "brand",
        "price"
    ]
].sort_values("event_time")

,event_time,event_type,product_id,brand,price
228967,2019-10-02 11:18:15 UTC,cart,5873241,lianail,6.33
232149,2019-10-02 11:35:12 UTC,remove_from_cart,5873241,lianail,6.33
232150,2019-10-02 11:35:12 UTC,remove_from_cart,5873241,lianail,6.33
232224,2019-10-02 11:35:37 UTC,remove_from_cart,5873241,lianail,6.33
232225,2019-10-02 11:35:37 UTC,remove_from_cart,5873241,lianail,6.33
232326,2019-10-02 11:36:02 UTC,remove_from_cart,5873241,lianail,6.33
232327,2019-10-02 11:36:02 UTC,remove_from_cart,5873241,lianail,6.33
232328,2019-10-02 11:36:02 UTC,remove_from_cart,5873241,lianail,6.33
232329,2019-10-02 11:36:02 UTC,remove_from_cart,5873241,lianail,6.33
232337,2019-10-02 11:36:03 UTC,remove_from_cart,5873241,lianail,6.33


### Exact Duplicate Assessment

Approximately 1.1 million exact duplicate records were detected within processing chunks. Most of these records were concentrated in `remove_from_cart` events.

Further inspection showed that identical `remove_from_cart` events can occur multiple times for the same user, product, session, and timestamp. In one inspected case, the same event appeared up to 14 times within a single second.

However, the dataset does not include quantity information or a unique event identifier that would allow these records to be reliably classified as technical duplicates. Therefore, exact equality alone is insufficient evidence that a repeated record is erroneous.

**Cleaning decision:**

- Do not automatically remove exact duplicate records.
- Preserve repeated events to avoid potentially deleting legitimate user interactions.
- Treat duplicate-sensitive event counts with caution during behavioral analysis.
- Document this limitation when interpreting event-based funnel metrics.


In [24]:
missing_values = pd.Series ( 
    0,
    index=["category_code", "brand", "user_session"],
    dtype="int64"
 )
total_rows = 0

In [25]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["category_code", "brand", "user_session"]
    ):
        missing_values += chunk[
            ["category_code", "brand", "user_session"]
        ].isna().sum()

        total_rows += len(chunk)

In [26]:
missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": (
        missing_values / total_rows * 100
    )
})

missing_summary

,missing_count,missing_percentage
category_code,20339246,98.291225
brand,8757117,42.319551
user_session,4598,0.022220


I would not remove rows with missing category_code values because 98.29% of the records are missing this field. Dropping these rows would result in a substantial loss of valuable behavioral data. Since category_id is still available, it can be used to preserve category-level information.

In [27]:
products_with_brand  = set()
products_without_brand = set()

In [28]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["product_id", "brand"]
    ):
        products_with_brand.update(
            chunk.loc[
                chunk["brand"].notna(),
                "product_id"
            ].unique()
        )

        products_without_brand.update(
            chunk.loc[
                chunk["brand"].isna(),
                "product_id"
            ].unique()
        )


In [29]:
products_mixed_brand = (
    products_with_brand
    .intersection(products_without_brand)
)

print("Products with known brand:", len(products_with_brand))
print("Products with missing brand:", len(products_without_brand))
print("Products appearing both with and without brand:", len(products_mixed_brand))

Products with known brand: 31098
Products with missing brand: 35726
Products appearing both with and without brand: 12253


In [30]:
product_brand_pairs = set()

In [31]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["product_id", "brand"]
    ):
        known_brand_rows = chunk[
            chunk["brand"].notna()
        ][["product_id", "brand"]].drop_duplicates()

        product_brand_pairs.update(
            known_brand_rows.itertuples(
                index=False,
                name=None
            )
        )

In [32]:
product_brand_df = pd.DataFrame(
    product_brand_pairs,
    columns=["product_id", "brand"]
)

brands_per_product = (
    product_brand_df
    .groupby("product_id")["brand"]
    .nunique()
)

print(
    "Products with exactly one known brand:",
    (brands_per_product == 1).sum()
)

print(
    "Products with multiple known brands:",
    (brands_per_product > 1).sum()
)

Products with exactly one known brand: 31074
Products with multiple known brands: 24


In [33]:
single_brand_products = brands_per_product[
    brands_per_product == 1
].index

safe_brand_map = (
    product_brand_df[
        product_brand_df["product_id"].isin(single_brand_products)
    ]
    .set_index("product_id")["brand"]
    .to_dict()
)

len(safe_brand_map)

31074

In [34]:
recoverable_brand_values = 0

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["product_id", "brand"]
    ):
        recoverable_mask = (
            chunk["brand"].isna()
            & chunk["product_id"].isin(safe_brand_map)
        )

        recoverable_brand_values += recoverable_mask.sum()

print("Missing brand values:", int(missing_values["brand"]))
print("Recoverable brand values:", int(recoverable_brand_values))

Missing brand values: 8757117
Recoverable brand values: 98300


I would recover the 98,300 missing brand values because we have reliable evidence to infer them from product_id. However, from a business perspective, I would first evaluate whether the expected analytical benefit justifies the additional time and cost. These records represent only about 1.12% of all missing brand values, so the impact should be assessed before investing additional resources.

In [35]:
recoverable_brand_percentage = (
    recoverable_brand_values
    / missing_values["brand"]
    * 100
)

print(
    f"Recoverable missing brand values: "
    f"{recoverable_brand_percentage:.2f}%"
)

Recoverable missing brand values: 1.12%


In [36]:
test_brand_chunk = pd.read_csv(
    csv_files[0],
    nrows=500000
)

missing_before = test_brand_chunk["brand"].isna().sum()

test_brand_chunk["brand"] = (
    test_brand_chunk["brand"]
    .fillna(
        test_brand_chunk["product_id"].map(safe_brand_map)
    )
)

missing_after = test_brand_chunk["brand"].isna().sum()

print("Missing brand before:", missing_before)
print("Missing brand after:", missing_after)
print("Recovered:", missing_before - missing_after)

Missing brand before: 197261
Missing brand after: 194796
Recovered: 2465


In [37]:
chunk["brand"] = (
    chunk["brand"]
    .fillna(
        chunk["product_id"].map(safe_brand_map)
    )
)

Recover missing brand values only when the corresponding product_id has exactly one consistently observed brand. Preserve all remaining missing brand values rather than assigning unsupported information.

In [38]:
missing_session_events = pd.Series(dtype="int64")

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["event_type", "user_session"]
    ):
        missing_sessions = chunk[
            chunk["user_session"].isna()
        ]

        event_counts = (
            missing_sessions["event_type"]
            .value_counts()
        )

        missing_session_events = (
            missing_session_events
            .add(event_counts, fill_value=0)
        )

missing_session_events.sort_values(ascending=False)

event_type
cart                3776.0
remove_from_cart     791.0
view                  31.0
dtype: float64

In [39]:
users_missing_session = set()

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["user_id", "user_session"]
    ):
        missing_mask = chunk["user_session"].isna()

        users_missing_session.update(
            chunk.loc[
                missing_mask,
                "user_id"
            ].unique()
        )

print(
    "Unique users with missing sessions:",
    len(users_missing_session)
)

Unique users with missing sessions: 1533


In [40]:
users_with_known_session = set()

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["user_id", "user_session"]
    ):
        known_mask = (
            chunk["user_session"].notna()
            & chunk["user_id"].isin(users_missing_session)
        )

        users_with_known_session.update(
            chunk.loc[
                known_mask,
                "user_id"
            ].unique()
        )

print(
    "Users with missing sessions:",
    len(users_missing_session)
)

print(
    "Users also appearing with known sessions:",
    len(users_with_known_session)
)

Users with missing sessions: 1533
Users also appearing with known sessions: 1326


In [41]:
missing_session_user_events = []

In [42]:
for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=[
            "event_time",
            "event_type",
            "product_id",
            "user_id",
            "user_session"
        ]
    ):
        affected_users_mask = (
            chunk["user_id"].isin(users_missing_session)
        )

        affected_events = chunk.loc[
            affected_users_mask
        ].copy()

        if not affected_events.empty:
            missing_session_user_events.append(
                affected_events
            )

In [43]:
session_context_df = pd.concat(
    missing_session_user_events,
    ignore_index=True
)

session_context_df["event_time"] = pd.to_datetime(
    session_context_df["event_time"],
    utc=True
)

print("Context rows:", len(session_context_df))
print(
    "Missing sessions:",
    session_context_df["user_session"].isna().sum()
)

Context rows: 40733
Missing sessions: 4598


In [44]:
session_context_df = (
    session_context_df
    .sort_values(
        ["user_id", "event_time"]
    )
    .reset_index(drop=True)
)

session_context_df.head(10)

,event_time,event_type,product_id,user_id,user_session
0,2019-11-06 15:12:02+00:00,remove_from_cart,5660213,230956608,NaN
1,2019-11-06 15:12:04+00:00,remove_from_cart,5773401,230956608,NaN
2,2019-11-06 15:16:20+00:00,cart,5856952,230956608,2330be16-75f0-4fa1-926a-26fc7189dfcc
3,2019-11-06 15:17:46+00:00,view,5886199,230956608,5a5ec2f2-e2b3-4e0e-8d73-4fd0c53cc852
4,2019-11-06 15:18:02+00:00,view,5831648,230956608,5a5ec2f2-e2b3-4e0e-8d73-4fd0c53cc852
5,2019-11-06 15:18:42+00:00,view,5742718,230956608,5a5ec2f2-e2b3-4e0e-8d73-4fd0c53cc852
6,2019-11-06 15:19:52+00:00,cart,5886199,230956608,5a5ec2f2-e2b3-4e0e-8d73-4fd0c53cc852
7,2019-11-06 15:20:36+00:00,cart,5628025,230956608,5a5ec2f2-e2b3-4e0e-8d73-4fd0c53cc852
8,2019-11-06 15:23:31+00:00,cart,5837101,230956608,5a5ec2f2-e2b3-4e0e-8d73-4fd0c53cc852
9,2019-11-06 15:26:16+00:00,cart,5665851,230956608,5a5ec2f2-e2b3-4e0e-8d73-4fd0c53cc852


In [45]:
session_context_df["previous_session"] = (
    session_context_df
    .groupby("user_id")["user_session"]
    .ffill()
)

session_context_df["next_session"] = (
    session_context_df
    .groupby("user_id")["user_session"]
    .bfill()
)

In [46]:
missing_session_analysis = session_context_df[
    session_context_df["user_session"].isna()
].copy()

same_surrounding_session = (
    missing_session_analysis["previous_session"].notna()
    & missing_session_analysis["next_session"].notna()
    & (
        missing_session_analysis["previous_session"]
        == missing_session_analysis["next_session"]
    )
)

print(
    "Missing sessions:",
    len(missing_session_analysis)
)

print(
    "Same previous and next session:",
    same_surrounding_session.sum()
    
)

Missing sessions: 4598
Same previous and next session: 52


In [47]:
session_context_df["previous_event_time"] = (
    session_context_df
    .groupby("user_id")["event_time"]
    .shift(1)
)

session_context_df["next_event_time"] = (
    session_context_df
    .groupby("user_id")["event_time"]
    .shift(-1)
)

In [48]:
session_context_df["previous_known_time"] = (
    session_context_df["event_time"]
    .where(session_context_df["user_session"].notna())
    .groupby(session_context_df["user_id"])
    .ffill()
)

session_context_df["next_known_time"] = (
    session_context_df["event_time"]
    .where(session_context_df["user_session"].notna())
    .groupby(session_context_df["user_id"])
    .bfill()
)

In [49]:
session_candidates = session_context_df[
    session_context_df["user_session"].isna()
    & session_context_df["previous_session"].notna()
    & session_context_df["next_session"].notna()
    & (
        session_context_df["previous_session"]
        == session_context_df["next_session"]
    )
].copy()

session_candidates["minutes_from_previous"] = (
    session_candidates["event_time"]
    - session_candidates["previous_known_time"]
).dt.total_seconds() / 60

session_candidates["minutes_to_next"] = (
    session_candidates["next_known_time"]
    - session_candidates["event_time"]
).dt.total_seconds() / 60

session_candidates[
    [
        "event_time",
        "event_type",
        "user_id",
        "previous_session",
        "next_session",
        "minutes_from_previous",
        "minutes_to_next"
    ]
].head(20)

,event_time,event_type,user_id,previous_session,next_session,minutes_from_previous,minutes_to_next
1947,2019-12-02 14:39:59+00:00,cart,366954205,f1efd2b4-26f0-4606-a54b-c38680d6d0ee,f1efd2b4-26f0-4606-a54b-c38680d6d0ee,1.650000,247.366667
1948,2019-12-02 14:40:07+00:00,cart,366954205,f1efd2b4-26f0-4606-a54b-c38680d6d0ee,f1efd2b4-26f0-4606-a54b-c38680d6d0ee,1.783333,247.233333
1949,2019-12-02 14:40:50+00:00,cart,366954205,f1efd2b4-26f0-4606-a54b-c38680d6d0ee,f1efd2b4-26f0-4606-a54b-c38680d6d0ee,2.500000,246.516667
4506,2020-01-19 11:31:27+00:00,cart,431452539,3e3756cb-b9f4-4933-a84f-2fea429f9527,3e3756cb-b9f4-4933-a84f-2fea429f9527,0.350000,195.350000
4507,2020-01-19 11:31:27+00:00,cart,431452539,3e3756cb-b9f4-4933-a84f-2fea429f9527,3e3756cb-b9f4-4933-a84f-2fea429f9527,0.350000,195.350000
4508,2020-01-19 11:31:30+00:00,cart,431452539,3e3756cb-b9f4-4933-a84f-2fea429f9527,3e3756cb-b9f4-4933-a84f-2fea429f9527,0.400000,195.300000
4509,2020-01-19 11:31:32+00:00,cart,431452539,3e3756cb-b9f4-4933-a84f-2fea429f9527,3e3756cb-b9f4-4933-a84f-2fea429f9527,0.433333,195.266667
4510,2020-01-19 11:31:32+00:00,cart,431452539,3e3756cb-b9f4-4933-a84f-2fea429f9527,3e3756cb-b9f4-4933-a84f-2fea429f9527,0.433333,195.266667
4511,2020-01-19 11:31:34+00:00,cart,431452539,3e3756cb-b9f4-4933-a84f-2fea429f9527,3e3756cb-b9f4-4933-a84f-2fea429f9527,0.466667,195.233333
4512,2020-01-19 11:31:36+00:00,cart,431452539,3e3756cb-b9f4-4933-a84f-2fea429f9527,3e3756cb-b9f4-4933-a84f-2fea429f9527,0.500000,195.200000


In [50]:
session_candidates[
    [
        "minutes_from_previous",
        "minutes_to_next"
    ]
].describe()


,minutes_from_previous,minutes_to_next
count,52.000000,52.000000
mean,0.667628,212.319231
std,0.552114,308.553439
min,0.083333,0.066667
25%,0.337500,1.408333
50%,0.516667,194.958333
75%,0.766667,195.275000
max,2.500000,1676.316667


### Missing Value Treatment

Missing values were evaluated separately according to their analytical role and the available evidence.

#### `category_code`

Approximately 98.29% of records have a missing `category_code`. Removing these records would result in substantial information loss. Since `category_id` remains available, missing `category_code` values will be preserved rather than imputed or used as a reason to remove records.

#### `brand`

Approximately 42.32% of records have a missing `brand`. Product-brand relationships were analyzed to determine whether some missing values could be recovered.

Among products with known brand information, 31,074 product IDs were consistently associated with exactly one brand, while only 24 were associated with multiple brands. Using the consistent product-to-brand relationships, 98,300 missing brand values (approximately 1.12% of all missing brand values) can be recovered.

**Decision:** Missing `brand` values will only be imputed when the corresponding `product_id` has exactly one consistently observed brand. All remaining missing brand values will be preserved.

#### `user_session`

Only 4,598 records (approximately 0.022% of the dataset) have a missing `user_session`. These records consist primarily of `cart` and `remove_from_cart` events, with no purchase events affected.

Temporal analysis identified only 52 missing-session events surrounded by the same known session identifier. Although these events were generally close to the preceding known session, temporal proximity alone is insufficient evidence to reconstruct a session identifier reliably.

**Decision:** Missing `user_session` values will be preserved. Records without a session identifier may remain available for event-, user-, and product-level analyses but will be excluded from metrics that explicitly require a known session.

In [51]:
test_brand_chunk["price_quality"] = np.select(
    [
        test_brand_chunk["price"] < 0,
        test_brand_chunk["price"] == 0
    ],
    [
        "negative",
        "zero"
    ],
    default="positive"
)

test_brand_chunk[
    ["price", "price_quality"]
].head(10)

,price,price_quality
0,2.62,positive
1,2.62,positive
2,13.48,positive
3,2.62,positive
4,0.56,positive
5,2.62,positive
6,4.75,positive
7,0.56,positive
8,1.27,positive
9,1.59,positive


In [52]:
test_brand_chunk["price_quality"].value_counts()

price_quality
positive    499339
zero           657
negative         4
Name: count, dtype: int64

I would preserve the event but treat its price differently when calculating revenue. Keeping negative values makes the analysis more complete because they may reveal anomalies or specific areas that require further investigation. I prefer having anomalous data that can be analyzed rather than deleting information without sufficient evidence.

Negative prices should be preserved and explicitly flagged rather than removed. However, they should not automatically be interpreted as losses or refunds because their business meaning is unknown.

In [53]:
test_brand_chunk["purchase_revenue"] = np.where(
    (test_brand_chunk["event_type"] == "purchase")
    & (test_brand_chunk["price"] > 0),
    test_brand_chunk["price"],
    0
)

In [54]:
test_brand_chunk["purchase_revenue"].sum()

np.float64(141411.99)

In [55]:
test_brand_chunk[
    (test_brand_chunk["event_type"] == "purchase")
    & (test_brand_chunk["price"] <= 0)
][
    [
        "event_type",
        "price",
        "price_quality",
        "purchase_revenue"
    ]
]

,event_type,price,price_quality,purchase_revenue
112860,purchase,-23.81,negative,0.0
198302,purchase,-7.94,negative,0.0
436918,purchase,-47.62,negative,0.0
443204,purchase,-15.87,negative,0.0


### Price Quality Treatment

Price anomalies identified during Data Understanding were preserved rather than removed because their underlying business meaning cannot be determined from the available data.

A `price_quality` feature was created to classify records as:

- `positive`: price greater than zero
- `zero`: price equal to zero
- `negative`: price below zero

The original `price` column is preserved without modification.

For revenue analysis, a separate `purchase_revenue` feature will be created. It records revenue only when the event is a `purchase` and the associated price is positive. Purchase events with zero or negative prices remain in the dataset for behavioral analysis but contribute zero to this positive purchase revenue metric.

Negative prices are not interpreted as refunds, losses, or adjustments because the dataset does not provide sufficient evidence to establish their business meaning.

**Cleaning decision:**

- Preserve all behavioral events regardless of price quality.
- Preserve the original `price`.
- Flag zero and negative prices through `price_quality`.
- Use positive-price purchases for the `purchase_revenue` analytical metric.
- Analyze anomalous prices separately when relevant.

In [56]:
print("PyArrow version:", pa.__version__)

PyArrow version: 25.0.1


In [57]:
test_parquet_path = processed_data_path / "test_clean.parquet"

test_brand_chunk.head(1000).to_parquet(
    test_parquet_path,
    index=False
)

print("File created:", test_parquet_path.exists())

File created: True


In [58]:
test_parquet_path.unlink()

print("Test file exists:", test_parquet_path.exists())

Test file exists: False


In [59]:
len(safe_brand_map)

31074

In [60]:
clean_data_path = processed_data_path / "ecommerce_events_clean"

clean_data_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Clean data directory:", clean_data_path.exists())

Clean data directory: True


In [61]:
month_order = {
    "2019-Oct.csv": 1,
    "2019-Nov.csv": 2,
    "2019-Dec.csv": 3,
    "2020-Jan.csv": 4,
    "2020-Feb.csv": 5
}

csv_files = sorted(
    csv_files,
    key=lambda file: month_order[file.name]
)

for file in csv_files:
    print(file.name)

2019-Oct.csv
2019-Nov.csv
2019-Dec.csv
2020-Jan.csv
2020-Feb.csv


In [62]:
for file in csv_files:
    output_file = clean_data_path / f"{file.stem}_clean.parquet"
    print(output_file.name)

2019-Oct_clean.parquet
2019-Nov_clean.parquet
2019-Dec_clean.parquet
2020-Jan_clean.parquet
2020-Feb_clean.parquet


In [63]:
pipeline_test = pd.read_csv(
    csv_files[0],
    nrows=500000
)

pipeline_test["event_time"] = pd.to_datetime(
    pipeline_test["event_time"],
    utc=True,
    errors="coerce"
)

pipeline_test["brand"] = (
    pipeline_test["brand"]
    .fillna(
        pipeline_test["product_id"].map(safe_brand_map)
    )
)

pipeline_test["price_quality"] = np.select(
    [
        pipeline_test["price"] < 0,
        pipeline_test["price"] == 0
    ],
    [
        "negative",
        "zero"
    ],
    default="positive"
)

pipeline_test["purchase_revenue"] = np.where(
    (pipeline_test["event_type"] == "purchase")
    & (pipeline_test["price"] > 0),
    pipeline_test["price"],
    0
)

In [64]:
pipeline_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype              
---  ------            --------------   -----              
 0   event_time        500000 non-null  datetime64[us, UTC]
 1   event_type        500000 non-null  str                
 2   product_id        500000 non-null  int64              
 3   category_id       500000 non-null  int64              
 4   category_code     7712 non-null    str                
 5   brand             305204 non-null  str                
 6   price             500000 non-null  float64            
 7   user_id           500000 non-null  int64              
 8   user_session      499954 non-null  str                
 9   price_quality     500000 non-null  str                
 10  purchase_revenue  500000 non-null  float64            
dtypes: datetime64[us, UTC](1), float64(2), int64(3), str(5)
memory usage: 68.1 MB


In [66]:
writer_test_path = clean_data_path / "writer_test.parquet"

table = pa.Table.from_pandas(
    pipeline_test,
    preserve_index=False
)

with pq.ParquetWriter(
    writer_test_path,
    table.schema
) as writer:
    writer.write_table(table)

print("Writer test created:", writer_test_path.exists())

Writer test created: True


In [67]:
writer_test_path.unlink()

print("Writer test exists:", writer_test_path.exists())

Writer test exists: False


In [70]:
pipeline_stats = {
    "input_rows": 0,
    "output_rows": 0,
    "brands_recovered": 0,
    "invalid_event_times": 0
}
pipeline_stats

{'input_rows': 0,
 'output_rows': 0,
 'brands_recovered': 0,
 'invalid_event_times': 0}

In [71]:
pipeline_stats["input_rows"] == pipeline_stats["output_rows"]

True

### Final Data Preparation Pipeline

The final cleaning pipeline processes the five monthly datasets in chunks to maintain memory efficiency.

The pipeline applies the following transformations:

- Converts `event_time` to a timezone-aware UTC datetime.
- Preserves all behavioral events, including exact duplicate-looking records, because the available data does not provide sufficient evidence to classify them as ingestion errors.
- Recovers missing `brand` values only when a product has exactly one known brand across the dataset.
- Preserves unresolved missing values in `brand`, `category_code`, and `user_session`.
- Creates `price_quality` to classify prices as positive, zero, or negative.
- Preserves the original `price` field for traceability.
- Creates `purchase_revenue`, where only positive-price purchase events contribute to the primary revenue metric.
- Preserves zero and negative-price events for separate behavioral and data-quality analysis.
- Writes the processed data to Parquet files for more efficient storage and downstream analysis.

Pipeline validation metrics are collected to verify row preservation, recovered brand values, and timestamp integrity.

In [72]:
file = csv_files[0]

output_file = clean_data_path / f"{file.stem}_clean.parquet"

print("Input:", file.name)
print("Output:", output_file.name)

Input: 2019-Oct.csv
Output: 2019-Oct_clean.parquet


In [73]:
oct_stats = {
    "input_rows": 0,
    "output_rows": 0,
    "brands_recovered": 0,
    "invalid_event_times": 0
}

oct_stats

{'input_rows': 0,
 'output_rows': 0,
 'brands_recovered': 0,
 'invalid_event_times': 0}

In [74]:
writer = None

In [75]:
if output_file.exists():
    output_file.unlink()
    print("Previous output removed.")
else:
    print("No previous output found.")

No previous output found.


In [76]:
for chunk in pd.read_csv(
    file,
    chunksize=500000
):
    # Count input rows
    oct_stats["input_rows"] += len(chunk)

    # Convert event_time to UTC datetime
    chunk["event_time"] = pd.to_datetime(
        chunk["event_time"],
        utc=True,
        errors="coerce"
    )

    # Count invalid timestamps
    oct_stats["invalid_event_times"] += (
        chunk["event_time"].isna().sum()
    )

    # Identify recoverable missing brands
    brand_missing_before = chunk["brand"].isna()

    recoverable_brand_mask = (
        brand_missing_before
        & chunk["product_id"].isin(safe_brand_map)
    )

    oct_stats["brands_recovered"] += (
        recoverable_brand_mask.sum()
    )

    # Recover safe brand values
    chunk["brand"] = chunk["brand"].fillna(
        chunk["product_id"].map(safe_brand_map)
    )

    # Classify price quality
    chunk["price_quality"] = np.select(
        [
            chunk["price"] < 0,
            chunk["price"] == 0
        ],
        [
            "negative",
            "zero"
        ],
        default="positive"
    )

    # Create analytical revenue field
    chunk["purchase_revenue"] = np.where(
        (chunk["event_type"] == "purchase")
        & (chunk["price"] > 0),
        chunk["price"],
        0
    )

In [77]:
oct_stats = {
    "input_rows": 0,
    "output_rows": 0,
    "brands_recovered": 0,
    "invalid_event_times": 0
}

writer = None

oct_stats

{'input_rows': 0,
 'output_rows': 0,
 'brands_recovered': 0,
 'invalid_event_times': 0}

In [78]:
for chunk in pd.read_csv(
    file,
    chunksize=500000
):
    # 1. Count input rows
    oct_stats["input_rows"] += len(chunk)

    # 2. Convert event_time to UTC datetime
    chunk["event_time"] = pd.to_datetime(
        chunk["event_time"],
        utc=True,
        errors="coerce"
    )

    # 3. Count invalid timestamps
    oct_stats["invalid_event_times"] += (
        chunk["event_time"].isna().sum()
    )

    # 4. Identify recoverable missing brands
    recoverable_brand_mask = (
        chunk["brand"].isna()
        & chunk["product_id"].isin(safe_brand_map)
    )

    oct_stats["brands_recovered"] += (
        recoverable_brand_mask.sum()
    )

    # 5. Recover brand when product_id has one known brand
    chunk["brand"] = chunk["brand"].fillna(
        chunk["product_id"].map(safe_brand_map)
    )

    # 6. Classify price quality
    chunk["price_quality"] = np.select(
        [
            chunk["price"] < 0,
            chunk["price"] == 0
        ],
        [
            "negative",
            "zero"
        ],
        default="positive"
    )

    # 7. Create primary revenue metric
    chunk["purchase_revenue"] = np.where(
        (chunk["event_type"] == "purchase")
        & (chunk["price"] > 0),
        chunk["price"],
        0
    )

    # 8. Convert Pandas DataFrame to PyArrow Table
    table = pa.Table.from_pandas(
        chunk,
        preserve_index=False
    )

    # 9. Create writer using the first chunk schema
    if writer is None:
        writer = pq.ParquetWriter(
            output_file,
            table.schema,
            compression="snappy"
        )

    # 10. Write current chunk
    writer.write_table(table)

    # 11. Count output rows
    oct_stats["output_rows"] += len(chunk)

# Close the Parquet file
if writer is not None:
    writer.close()

In [79]:
print("Output file exists:", output_file.exists())
print()
print("October pipeline statistics:")
print(oct_stats)
print()
print(
    "Input equals output:",
    oct_stats["input_rows"] == oct_stats["output_rows"]
)

Output file exists: True

October pipeline statistics:
{'input_rows': 4102283, 'output_rows': 4102283, 'brands_recovered': np.int64(20370), 'invalid_event_times': np.int64(0)}

Input equals output: True


In [80]:
oct_parquet = pq.ParquetFile(output_file)

print("Rows stored in Parquet:", oct_parquet.metadata.num_rows)
print("Columns stored:", oct_parquet.metadata.num_columns)
print("Row groups:", oct_parquet.metadata.num_row_groups)

Rows stored in Parquet: 4102283
Columns stored: 11
Row groups: 9


In [81]:
pipeline_stats["input_rows"] += oct_stats["input_rows"]
pipeline_stats["output_rows"] += oct_stats["output_rows"]
pipeline_stats["brands_recovered"] += int(oct_stats["brands_recovered"])
pipeline_stats["invalid_event_times"] += int(oct_stats["invalid_event_times"])

pipeline_stats

{'input_rows': 4102283,
 'output_rows': 4102283,
 'brands_recovered': 20370,
 'invalid_event_times': 0}

In [82]:
remaining_files = csv_files[1:]

for file in remaining_files:
    print(file.name)

2019-Nov.csv
2019-Dec.csv
2020-Jan.csv
2020-Feb.csv


In [83]:
monthly_stats = {}

monthly_stats

{}

In [84]:
def process_month(file, clean_data_path, safe_brand_map):
    output_file = clean_data_path / f"{file.stem}_clean.parquet"

    stats = {
        "input_rows": 0,
        "output_rows": 0,
        "brands_recovered": 0,
        "invalid_event_times": 0
    }

    writer = None

    if output_file.exists():
        output_file.unlink()

    for chunk in pd.read_csv(
        file,
        chunksize=500000
    ):
        # Count input rows
        stats["input_rows"] += len(chunk)

        # Convert timestamp
        chunk["event_time"] = pd.to_datetime(
            chunk["event_time"],
            utc=True,
            errors="coerce"
        )

        stats["invalid_event_times"] += int(
            chunk["event_time"].isna().sum()
        )

        # Identify safely recoverable brands
        recoverable_brand_mask = (
            chunk["brand"].isna()
            & chunk["product_id"].isin(safe_brand_map)
        )

        stats["brands_recovered"] += int(
            recoverable_brand_mask.sum()
        )

        # Recover brand
        chunk["brand"] = chunk["brand"].fillna(
            chunk["product_id"].map(safe_brand_map)
        )

        # Price-quality classification
        chunk["price_quality"] = np.select(
            [
                chunk["price"] < 0,
                chunk["price"] == 0
            ],
            [
                "negative",
                "zero"
            ],
            default="positive"
        )

        # Primary revenue metric
        chunk["purchase_revenue"] = np.where(
            (chunk["event_type"] == "purchase")
            & (chunk["price"] > 0),
            chunk["price"],
            0
        )

        # Convert to PyArrow
        table = pa.Table.from_pandas(
            chunk,
            preserve_index=False
        )

        # Initialize writer from first chunk
        if writer is None:
            writer = pq.ParquetWriter(
                output_file,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)

        stats["output_rows"] += len(chunk)

    if writer is not None:
        writer.close()

    return stats

In [85]:
nov_file = remaining_files[0]

nov_stats = process_month(
    nov_file,
    clean_data_path,
    safe_brand_map
)

nov_stats

{'input_rows': 4635837,
 'output_rows': 4635837,
 'brands_recovered': 24118,
 'invalid_event_times': 0}

In [86]:
nov_output_file = clean_data_path / "2019-Nov_clean.parquet"

nov_parquet = pq.ParquetFile(nov_output_file)

print("Output file exists:", nov_output_file.exists())
print("Rows stored in Parquet:", nov_parquet.metadata.num_rows)
print("Columns stored:", nov_parquet.metadata.num_columns)
print("Row groups:", nov_parquet.metadata.num_row_groups)
print(
    "Stored rows equal input:",
    nov_parquet.metadata.num_rows == nov_stats["input_rows"]
)

Output file exists: True
Rows stored in Parquet: 4635837
Columns stored: 11
Row groups: 10
Stored rows equal input: True


In [87]:
# Store November statistics
monthly_stats["2019-Nov"] = nov_stats

# Add November to global pipeline statistics
for key in pipeline_stats:
    pipeline_stats[key] += int(nov_stats[key])

print("Monthly statistics:")
print(monthly_stats)

print("\nGlobal pipeline statistics:")
print(pipeline_stats)

Monthly statistics:
{'2019-Nov': {'input_rows': 4635837, 'output_rows': 4635837, 'brands_recovered': 24118, 'invalid_event_times': 0}}

Global pipeline statistics:
{'input_rows': 8738120, 'output_rows': 8738120, 'brands_recovered': 44488, 'invalid_event_times': 0}


In [88]:
for file in remaining_files[1:]:
    print(f"Processing {file.name}...")

    stats = process_month(
        file,
        clean_data_path,
        safe_brand_map
    )

    month_name = file.stem
    monthly_stats[month_name] = stats

    for key in pipeline_stats:
        pipeline_stats[key] += int(stats[key])

    print(f"Completed {file.name}")
    print(stats)
    print("-" * 50)

Processing 2019-Dec.csv...
Completed 2019-Dec.csv
{'input_rows': 3533286, 'output_rows': 3533286, 'brands_recovered': 8380, 'invalid_event_times': 0}
--------------------------------------------------
Processing 2020-Jan.csv...
Completed 2020-Jan.csv
{'input_rows': 4264752, 'output_rows': 4264752, 'brands_recovered': 4614, 'invalid_event_times': 0}
--------------------------------------------------
Processing 2020-Feb.csv...
Completed 2020-Feb.csv
{'input_rows': 4156682, 'output_rows': 4156682, 'brands_recovered': 40818, 'invalid_event_times': 0}
--------------------------------------------------


In [89]:
print("FINAL PIPELINE STATISTICS")
print("-" * 40)

for key, value in pipeline_stats.items():
    print(f"{key}: {value:,}")

print()
print(
    "All input rows preserved:",
    pipeline_stats["input_rows"] == pipeline_stats["output_rows"]
)

print()
print("MONTHLY STATISTICS")

for month, stats in monthly_stats.items():
    print(month, stats)

FINAL PIPELINE STATISTICS
----------------------------------------
input_rows: 20,692,840
output_rows: 20,692,840
brands_recovered: 98,300
invalid_event_times: 0

All input rows preserved: True

MONTHLY STATISTICS
2019-Nov {'input_rows': 4635837, 'output_rows': 4635837, 'brands_recovered': 24118, 'invalid_event_times': 0}
2019-Dec {'input_rows': 3533286, 'output_rows': 3533286, 'brands_recovered': 8380, 'invalid_event_times': 0}
2020-Jan {'input_rows': 4264752, 'output_rows': 4264752, 'brands_recovered': 4614, 'invalid_event_times': 0}
2020-Feb {'input_rows': 4156682, 'output_rows': 4156682, 'brands_recovered': 40818, 'invalid_event_times': 0}


In [90]:
monthly_stats["2019-Oct"] = oct_stats

monthly_stats = dict(
    sorted(
        monthly_stats.items(),
        key=lambda item: month_order[f"{item[0]}.csv"]
    )
)

for month, stats in monthly_stats.items():
    print(month, stats)

2019-Oct {'input_rows': 4102283, 'output_rows': 4102283, 'brands_recovered': np.int64(20370), 'invalid_event_times': np.int64(0)}
2019-Nov {'input_rows': 4635837, 'output_rows': 4635837, 'brands_recovered': 24118, 'invalid_event_times': 0}
2019-Dec {'input_rows': 3533286, 'output_rows': 3533286, 'brands_recovered': 8380, 'invalid_event_times': 0}
2020-Jan {'input_rows': 4264752, 'output_rows': 4264752, 'brands_recovered': 4614, 'invalid_event_times': 0}
2020-Feb {'input_rows': 4156682, 'output_rows': 4156682, 'brands_recovered': 40818, 'invalid_event_times': 0}


In [91]:
print("FINAL PARQUET VALIDATION")
print("-" * 60)

for month, stats in monthly_stats.items():
    parquet_file = clean_data_path / f"{month}_clean.parquet"

    parquet_metadata = pq.ParquetFile(parquet_file).metadata

    stored_rows = parquet_metadata.num_rows
    expected_rows = stats["output_rows"]

    print(
        f"{month} | "
        f"rows: {stored_rows:,} | "
        f"expected: {expected_rows:,} | "
        f"valid: {stored_rows == expected_rows}"
    )

FINAL PARQUET VALIDATION
------------------------------------------------------------
2019-Oct | rows: 4,102,283 | expected: 4,102,283 | valid: True
2019-Nov | rows: 4,635,837 | expected: 4,635,837 | valid: True
2019-Dec | rows: 3,533,286 | expected: 3,533,286 | valid: True
2020-Jan | rows: 4,264,752 | expected: 4,264,752 | valid: True
2020-Feb | rows: 4,156,682 | expected: 4,156,682 | valid: True


## Data Preparation Summary

The raw e-commerce dataset contained 20,692,840 behavioral events across five monthly files covering October 2019 through February 2020.

The data preparation pipeline was designed to preserve behavioral information while improving data quality and analytical usability.

### Event Time

The `event_time` field was converted from string format to a timezone-aware UTC datetime representation.

- Invalid timestamps detected: **0**
- No records were removed due to timestamp quality issues.

### Brand Recovery

Missing brand values were investigated at the product level.

A brand value was recovered only when a `product_id` was consistently associated with exactly one known brand elsewhere in the dataset. Products associated with multiple known brands were excluded from this recovery process to avoid ambiguous imputations.

- Safely recoverable brand values: **98,300**
- Remaining missing brand values were preserved rather than filled with unsupported assumptions.

This approach prioritizes analytical reliability over maximizing completeness.

### Session Information

Missing `user_session` values were preserved.

The investigation showed that session IDs could not be reconstructed reliably from surrounding events. Therefore, these events remain available for user-, product-, and event-level analyses but should be treated carefully in session-based metrics.

### Price Quality

Original price values were preserved, including zero and negative values.

A new `price_quality` field classifies each event as:

- `positive`
- `zero`
- `negative`

This preserves potentially meaningful anomalies for further investigation instead of deleting information whose business meaning cannot be confirmed from the available data.

### Revenue Metric

A separate `purchase_revenue` field was created for the primary revenue analysis.

Revenue is recognized only when:

- `event_type == "purchase"`
- `price > 0`

Zero and negative prices remain available in the original `price` field and can be analyzed separately as data-quality or business anomalies.

### Processed Dataset

The cleaned dataset was stored as monthly Parquet files using Snappy compression to provide more efficient storage and downstream analytical processing.

Final validation confirmed:

- **20,692,840 input rows**
- **20,692,840 output rows**
- **98,300 brand values safely recovered**
- **0 invalid event timestamps**
- **100% of input rows preserved**

All five processed Parquet files were validated against their expected row counts.

In [92]:
oct_clean = pd.read_parquet(
    clean_data_path / "2019-Oct_clean.parquet"
)

duplicate_mask = oct_clean.duplicated(
    subset=[
        "event_time",
        "event_type",
        "product_id",
        "category_id",
        "category_code",
        "brand",
        "price",
        "user_id",
        "user_session"
    ],
    keep=False
)

oct_duplicates = oct_clean[duplicate_mask]

print("October rows:", f"{len(oct_clean):,}")
print("Rows involved in exact duplicates:", f"{duplicate_mask.sum():,}")
print(
    "Percentage:",
    f"{duplicate_mask.mean() * 100:.2f}%"
)

print("\nDuplicates by event type:")
print(
    oct_duplicates["event_type"]
    .value_counts()
)

October rows: 4,102,283
Rows involved in exact duplicates: 418,237
Percentage: 10.20%

Duplicates by event type:
event_type
remove_from_cart    363653
cart                 53670
purchase               686
view                   228
Name: count, dtype: int64


In [94]:
oct_duplicates[
    [
        "event_time",
        "event_type",
        "product_id",
        "user_id",
        "user_session",
        "price"
    ]
].sort_values(
    [
        "user_id",
        "user_session",
        "event_time",
        "product_id"
    ]
).head(30)

,event_time,event_type,product_id,user_id,user_session,price
137684,2019-10-01 22:05:27+00:00,remove_from_cart,5812138,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb,4.68
137685,2019-10-01 22:05:27+00:00,remove_from_cart,5812138,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb,4.68
137746,2019-10-01 22:06:07+00:00,remove_from_cart,5739036,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb,3.32
137748,2019-10-01 22:06:07+00:00,remove_from_cart,5739036,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb,3.32
266253,2019-10-02 14:43:55+00:00,remove_from_cart,5709126,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad,4.21
266256,2019-10-02 14:43:55+00:00,remove_from_cart,5709126,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad,4.21
266393,2019-10-02 14:44:45+00:00,remove_from_cart,5823668,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad,7.60
266396,2019-10-02 14:44:45+00:00,remove_from_cart,5823668,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad,7.60
3535082,2019-10-27 16:00:04+00:00,remove_from_cart,5807805,10280338,fbacb4a3-2141-4f43-b9e5-97c911c7e770,4.44
3535083,2019-10-27 16:00:04+00:00,remove_from_cart,5807805,10280338,fbacb4a3-2141-4f43-b9e5-97c911c7e770,4.44


The table shows that there is not enough evidence to conclude that the exact duplicate rows are data errors. Since this is event-level data, identical records may represent repeated customer actions rather than accidental duplicates. Therefore, removing them could eliminate valid behavioral information.

In [95]:
oct_duplicates[
    [
        "event_time",
        "event_type",
        "product_id",
        "user_id",
        "user_session",
        "price"
    ]
].sort_values(
    [
        "user_id",
        "user_session",
        "event_time",
        "product_id"
    ]
).head(50)

,event_time,event_type,product_id,user_id,user_session,price
137684,2019-10-01 22:05:27+00:00,remove_from_cart,5812138,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb,4.68
137685,2019-10-01 22:05:27+00:00,remove_from_cart,5812138,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb,4.68
137746,2019-10-01 22:06:07+00:00,remove_from_cart,5739036,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb,3.32
137748,2019-10-01 22:06:07+00:00,remove_from_cart,5739036,8846226,ddd23270-792c-49f3-9ae2-d86f5080a5eb,3.32
266253,2019-10-02 14:43:55+00:00,remove_from_cart,5709126,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad,4.21
266256,2019-10-02 14:43:55+00:00,remove_from_cart,5709126,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad,4.21
266393,2019-10-02 14:44:45+00:00,remove_from_cart,5823668,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad,7.60
266396,2019-10-02 14:44:45+00:00,remove_from_cart,5823668,8846226,f835c9a8-cb8a-4201-85d4-00565bcc84ad,7.60
3535082,2019-10-27 16:00:04+00:00,remove_from_cart,5807805,10280338,fbacb4a3-2141-4f43-b9e5-97c911c7e770,4.44
3535083,2019-10-27 16:00:04+00:00,remove_from_cart,5807805,10280338,fbacb4a3-2141-4f43-b9e5-97c911c7e770,4.44


In [96]:
oct_duplicates["event_type"].value_counts(normalize=True).mul(100).round(2)

event_type
remove_from_cart    86.95
cart                12.83
purchase             0.16
view                 0.05
Name: proportion, dtype: float64

The analysis shows that remove_from_cart is the event type most frequently associated with exact duplicate records. However, removing these records could be risky because customers may legitimately remove products from their carts multiple times during their journey. Therefore, the repeated events may reflect real customer behavior rather than data errors, and the available evidence is not sufficient to justify deleting them.

### Exact Duplicate Assessment

Exact duplicate records were identified during the Data Understanding stage and were investigated before applying any removal strategy.

In the October dataset, 418,237 rows (10.20% of records) were involved in exact duplicate groups. The distribution of these records was highly concentrated in customer cart interactions:

- `remove_from_cart`: 86.95%
- `cart`: 12.83%
- `purchase`: 0.16%
- `view`: 0.05%

Manual inspection showed that identical records can occur for the same user, product, session, timestamp, event type, and price. However, the dataset does not contain sufficient information to determine whether these records represent ingestion duplicates or legitimate repeated customer actions.

This distinction is particularly important for behavioral event data because repeated `cart`, `remove_from_cart`, and even `purchase` events may contain meaningful information about the customer journey. Removing them without sufficient evidence could alter funnel behavior, purchase counts, and revenue-related analysis.

**Cleaning decision:**

- Preserve exact duplicate records in the processed dataset.
- Do not assume that identical event records are data errors.
- Document duplicate behavior as a data-quality limitation.
- Reassess duplicated events when constructing specific funnel or revenue metrics if their interpretation materially affects the analysis.

This approach prioritizes preservation of potentially meaningful behavioral information over unsupported deletion.

In [97]:
category_mapping_check = {}

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["category_id", "category_code"]
    ):
        known_categories = chunk.dropna(subset=["category_code"])

        for category_id, group in known_categories.groupby("category_id"):
            codes = set(group["category_code"].unique())

            if category_id not in category_mapping_check:
                category_mapping_check[category_id] = set()

            category_mapping_check[category_id].update(codes)

In [98]:
safe_category_ids = {
    category_id: next(iter(codes))
    for category_id, codes in category_mapping_check.items()
    if len(codes) == 1
}

ambiguous_category_ids = {
    category_id: codes
    for category_id, codes in category_mapping_check.items()
    if len(codes) > 1
}

print(
    "Category IDs with exactly one known category_code:",
    len(safe_category_ids)
)

print(
    "Category IDs with multiple known category_codes:",
    len(ambiguous_category_ids)
)

Category IDs with exactly one known category_code: 18
Category IDs with multiple known category_codes: 0


In [99]:
recoverable_category_codes = 0
total_missing_category_codes = 0

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        chunksize=500000,
        usecols=["category_id", "category_code"]
    ):
        missing_mask = chunk["category_code"].isna()

        total_missing_category_codes += missing_mask.sum()

        recoverable_mask = (
            missing_mask
            & chunk["category_id"].isin(safe_category_ids)
        )

        recoverable_category_codes += recoverable_mask.sum()

print(
    "Missing category_code values:",
    f"{total_missing_category_codes:,}"
)

print(
    "Safely recoverable category_code values:",
    f"{recoverable_category_codes:,}"
)

print(
    "Recoverable percentage:",
    f"{recoverable_category_codes / total_missing_category_codes * 100:.2f}%"
)

Missing category_code values: 20,339,246
Safely recoverable category_code values: 0
Recoverable percentage: 0.00%


The analysis shows that 20,339,246 records have missing category_code values, and none of these values can be safely recovered using the existing category mappings. Removing these records would result in a substantial loss of valuable customer interaction data. Therefore, missing category_code values should be preserved, while category_id remains the primary identifier for category-level analysis.

### Category Code Missing Values

The dataset contains 20,339,246 records with missing
`category_code` values.

An investigation of category mappings identified 18
category IDs with unique category codes and no
ambiguous mappings.

However, none of the missing values could be safely
recovered using the available mappings.

**Cleaning decision:**
- Preserve missing `category_code` values.
- Retain `category_id` for category-level analysis.
- Avoid unsupported category imputations.
- Preserve all customer interaction records.